# ADE-Sentinel - Kaggle run template

Step 0.5 of `PLAN.md`. Every neural run starts from a copy of this notebook.

**Before running:** in the right-hand panel set

| Setting | Value |
|---|---|
| Accelerator | `GPU T4 x2` for training; **`None`** for the PubMed fetch and the gensim jobs - both are CPU-only and a GPU just burns quota |
| Internet | **On** (repo clone, HF downloads, PubMed fetch) |
| Persistence | Save what matters to `/kaggle/working` before the session closes |

**Exit criterion for step 0.5:** cell 4 prints the split row counts and cell 5
prints `torch.cuda.device_count()`.


## 1. Pin what you install; record what you don't

Kaggle's base image ships torch/transformers/datasets and updates them without
warning, so reinstalling torch here is slow and can break CUDA. Install only
GROUP A from `requirements-remote.txt`, and *record* GROUP B (PLAN F10).

Skip this cell entirely for the fetch job - it needs nothing but `requests`.


In [ ]:
!pip install -q gensim==4.4.0 seqeval==1.2.2 pytorch-crf==0.7.2

import torch, transformers, datasets, numpy, sklearn

ENV = {
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'datasets': datasets.__version__,
    'numpy': numpy.__version__,
    'sklearn': sklearn.__version__,
}
print(ENV)


## 2. Get the code

The frozen splits are committed to git (PLAN F7), so a clone is all that is
needed for Stage 1 and Stage 2 data. Large artefacts - the PubMed corpus and the
embedding matrices - arrive separately as an attached Kaggle Dataset.

If the repo is private, add a GitHub token as the Kaggle secret `GITHUB_TOKEN`
(Add-ons -> Secrets) and the cell below will use it. Never paste a token inline:
this notebook is committed.


In [1]:
import sys, subprocess, pathlib, os

REPO_URL = 'https://github.com/sifatul-islam-onik/ADE-Sentinel.git'
REPO = pathlib.Path('/kaggle/working/ADE-Sentinel')

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass          # public repo, or the secret is not set - clone anonymously

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', url, str(REPO)], check=True)

sys.path.insert(0, str(REPO))
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo at', GIT_COMMIT)


repo at 42578b8


## 3. Attach the artefacts dataset

Right-hand panel -> **Add Input** -> search your private
`ade-sentinel-artifacts` dataset (published by `scripts/push_to_kaggle.py`).

**Record the version number it shows.** It goes into every run row, and it is
what makes a result reproducible after the dataset changes.


In [ ]:
DATASET_VERSION = 'v1'   # <-- read this off the Add Input panel, every session

ARTIFACTS = pathlib.Path('/kaggle/input/ade-sentinel-artifacts')
print('attached:', ARTIFACTS.exists())
if ARTIFACTS.exists():
    for p in sorted(ARTIFACTS.iterdir()):
        print(' ', p.name)


## 4. Verify the frozen splits

Reads what Phase 1.3 committed. Row counts here must match
`results/dataset_stats.md` exactly - if they do not, the notebook is running
against a stale clone.


In [ ]:
import pandas as pd

SPLITS = REPO / 'data' / 'splits'
found = sorted(SPLITS.glob('*.parquet'))

if not found:
    print('No splits yet - Phase 1.3 has not run. Expected after step 1.3.')
else:
    for f in found:
        df = pd.read_parquet(f)
        print(f'{f.name:34s} {len(df):>7,} rows  cols={list(df.columns)}')


## 5. Device count - read this before every training run

`Trainer` wraps the model in `DataParallel` when two GPUs are visible and
multiplies `per_device_train_batch_size` by the device count. Two consequences
(PLAN F8):

1. The PRD's **batch 16 at lr 2e-5** means `per_device_train_batch_size=8` here.
2. **Runs 3-6 must all see the same device count.** They are supposed to differ
   only in the embedding matrix; if one runs on 1 GPU and another on 2, the
   effective batch differs too and the headline ablation is contaminated.

The BiLSTM-CRF (run 10) must use a single GPU regardless: the CRF computes loss
inside `forward`, so `DataParallel` returns a per-GPU loss vector.


In [ ]:
N_GPU = torch.cuda.device_count()
print('device_count :', N_GPU)
for i in range(N_GPU):
    print(' ', i, torch.cuda.get_device_name(i))

PER_DEVICE_BATCH = 8
print('effective batch:', PER_DEVICE_BATCH * max(N_GPU, 1))

# Uncomment to force one GPU (runs 3-6 pinning, and run 10 CRF):
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # must precede the first torch CUDA call


## 6. Log the run

`log_run` derives `effective_batch` from the device count and stamps the git
commit, so the row describes the run rather than the intent.

`results/runs.csv` lives inside the clone, which **disappears when the session
ends**. Download it, or commit it, before you close the notebook.


In [ ]:
from src.utils import set_seed, log_run

SEED = set_seed(42)

# Template - fill in after a real run:
# log_run(
#     run_id=3, stage='1', model='bilstm', embedding='E0-random',
#     metrics={'macro_f1': 0.0, 'f1_pos': 0.0, 'f1_neg': 0.0, 'pr_auc': 0.0},
#     params={'hidden': 256, 'dropout': 0.5, 'max_len': 128, **ENV},
#     seed=SEED, dataset_version=DATASET_VERSION,
#     per_device_batch=PER_DEVICE_BATCH, epochs=5, lr=1e-3,
# )

print('ready. seed =', SEED)


---

# Step 0.6 / 1.4 - the PubMed fetch

**Run this job on its own, with Accelerator = `None`.** It is pure I/O; a GPU
session would spend your 30h/week quota on network waits.

Set the NCBI key as a Kaggle secret named `NCBI_API_KEY`
(**Add-ons -> Secrets -> Add a secret**). It raises the rate limit from 3 to 10
requests/second, which is the difference between roughly 3 hours and roughly 1.

The key must live in Secrets, never in this notebook - the notebook is committed
to git.


In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['NCBI_API_KEY'] = UserSecretsClient().get_secret('NCBI_API_KEY')
    print('NCBI key loaded from Kaggle Secrets -> 10 req/s')
except Exception as exc:
    print('No NCBI key; falling back to 3 req/s. Fetch will take ~3x longer.')
    print(' ', exc)


No NCBI key; falling back to 3 req/s. Fetch will take ~3x longer.
  No module named 'kaggle_secrets'


In [ ]:
!cd /kaggle/working/ADE-Sentinel && python scripts/fetch_pubmed.py \
    --out /kaggle/working/pubmed_corpus.jsonl


Then **save the corpus out of the session** - it is gone when the notebook stops:

- quickest: File -> Download `/kaggle/working/pubmed_corpus.jsonl`, then run
  `scripts/push_to_kaggle.py --init` locally, or
- in-session: **Save Version** with output, then Add Input it into later notebooks.

Record the printed record count and wall-clock time in
`report/data_documentation.md` (step 1.5).
